In [1]:
import pandas as pd

# Replace 'file_path.xlsx' with the path to your Excel file
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7.xlsx"

# Read the Excel file into a DataFrame
df = pd.read_excel(file_path)

# Display the first few rows of the DataFrame
df.head()

,db_id,spider_query,question,text2sql_query
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;"
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;"
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""..."
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O..."
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...


In [2]:
df.shape

(719, 4)

In [3]:
def calculate_accuracy(results):
    return (results.count(True) / len(results))*100

## Approach 1:Edit Distance

In [3]:
#!pip install Levenshtein

In [4]:
import Levenshtein

def normalized_edit_distance(str1, str2):
    """
    Calculate the normalized edit distance between two strings.
    
    Args:
        str1 (str): The first string.
        str2 (str): The second string.

    Returns:
        float: The normalized edit distance considering string lengths.
    """
    # Calculate raw edit distance using Levenshtein
    edit_distance = Levenshtein.distance(str1, str2)
    
    # Normalize by average string length
    avg_length = (len(str1) + len(str2)) / 2
    normalized_distance = edit_distance / avg_length if avg_length > 0 else 0
    
    return normalized_distance

In [5]:
def edit_dist_match(query1,query2):
    query1 = query1.lower()
    query2 = query2.lower()
    ed = normalized_edit_distance(query1, query2)
    print(f"Normalized Edit Distance Similarity: {ed}")
    return ed   

In [6]:
query1 = "SELECT COUNT(*) FROM farm;"
query2 = """SELECT f."Farm_ID", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;"""
edit_dist_match(query1,query2)

Normalized Edit Distance Similarity: 1.158878504672897


1.158878504672897

In [86]:
query1 = "SELECT COUNT(*) FROM farm;"
query2 = "SELECT count(*) FROM farm f;"
edit_dist_match(query1,query2)

Normalized Edit Distance Similarity: 0.07407407407407407


0.07407407407407407

In [87]:
query1 = """SELECT lname ,  sex FROM Student WHERE StuID IN (SELECT T1.StuID FROM Plays_games AS T1 JOIN Video_games AS T2 ON T1.GameID  =  T2.GameID WHERE T2.Gname  =  "Call of Destiny" INTERSECT SELECT T1.StuID FROM Plays_games AS T1 JOIN Video_games AS T2 ON T1.GameID  =  T2.GameID WHERE T2.Gname  =  "Works of Widenius")"""
query2 = """SELECT s.LastName, s.Gender FROM Student s JOIN Plays_Games pg ON s.StuID = pg.StuID JOIN Video_Games vg ON pg.GameID = vg.GameID WHERE vg.GName = 'Call of Destiny' AND vg.GName = 'Works of Widenius' GROUP BY s.LastName, s.Gender HAVING COUNT(DISTINCT vg.GName) = 2;"""
edit_dist_match(query1,query2)

Normalized Edit Distance Similarity: 0.6252158894645942


0.6252158894645942

In [7]:
edit_dist_result = []
for index, row in df.iterrows():
    spider_query = row['spider_query']
    text2sql_query = row['text2sql_query']
    res = edit_dist_match(spider_query,text2sql_query)
    edit_dist_result.append(res)

Normalized Edit Distance Similarity: 0.6388888888888888
Normalized Edit Distance Similarity: 0.6756756756756757
Normalized Edit Distance Similarity: 0.4931506849315068
Normalized Edit Distance Similarity: 0.28125
Normalized Edit Distance Similarity: 0.3111111111111111
Normalized Edit Distance Similarity: 0.19047619047619047
Normalized Edit Distance Similarity: 0.3089430894308943
Normalized Edit Distance Similarity: 0.4122137404580153
Normalized Edit Distance Similarity: 0.44025157232704404
Normalized Edit Distance Similarity: 0.44025157232704404
Normalized Edit Distance Similarity: 0.5607476635514018
Normalized Edit Distance Similarity: 0.5607476635514018
Normalized Edit Distance Similarity: 0.16470588235294117
Normalized Edit Distance Similarity: 0.3516483516483517
Normalized Edit Distance Similarity: 0.32061068702290074
Normalized Edit Distance Similarity: 0.32061068702290074
Normalized Edit Distance Similarity: 0.13071895424836602
Normalized Edit Distance Similarity: 0.1307189542483

In [8]:
len(edit_dist_result)

719

In [9]:
df['edit_distance'] = edit_dist_result

## Approach 2 : Embedding Matching

In [10]:
from sentence_transformers import SentenceTransformer, util
# Load a pre-trained model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')

C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\t

In [11]:
def embedding_match(query1,query2):
    query1 = query1.lower()
    query2 = query2.lower()
    # Compute embeddings
    embedding1 = model.encode(query1)
    embedding2 = model.encode(query2)
    # Compute cosine similarity
    similarity = util.cos_sim(embedding1, embedding2)
    print(f"Semantic Similarity: {similarity.item()}")
    return similarity.item()

In [94]:
query1 = "SELECT COUNT(*) FROM farm;"
query2 = """SELECT f."Farm_ID", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;"""
embedding_match(query1,query2)

Semantic Similarity: 0.5847867131233215


0.5847867131233215

In [95]:
# Example queries
query1 = "SELECT COUNT(*) FROM farm;"
query2 = "SELECT count(*) FROM farm f;"
embedding_match(query1,query2)

Semantic Similarity: 0.9586180448532104


0.9586180448532104

In [96]:
query1 = """SELECT lname ,  sex FROM Student WHERE StuID IN (SELECT T1.StuID FROM Plays_games AS T1 JOIN Video_games AS T2 ON T1.GameID  =  T2.GameID WHERE T2.Gname  =  "Call of Destiny" INTERSECT SELECT T1.StuID FROM Plays_games AS T1 JOIN Video_games AS T2 ON T1.GameID  =  T2.GameID WHERE T2.Gname  =  "Works of Widenius")"""
query2 = """SELECT s.LastName, s.Gender FROM Student s JOIN Plays_Games pg ON s.StuID = pg.StuID JOIN Video_Games vg ON pg.GameID = vg.GameID WHERE vg.GName = 'Call of Destiny' AND vg.GName = 'Works of Widenius' GROUP BY s.LastName, s.Gender HAVING COUNT(DISTINCT vg.GName) = 2;"""
embedding_match(query1,query2)

Semantic Similarity: 0.8142150640487671


0.8142150640487671

In [12]:
embed_result = []
i = 0
for index, row in df.iterrows():
    i = i+1
    print(i)
    spider_query = row['spider_query']
    text2sql_query = row['text2sql_query']
    res = embedding_match(spider_query,text2sql_query)
    embed_result.append(res)

1
Semantic Similarity: 0.7701829671859741
2
Semantic Similarity: 0.7547934651374817
3
Semantic Similarity: 0.8307386636734009
4
Semantic Similarity: 0.8872911930084229
5
Semantic Similarity: 0.8674377202987671
6
Semantic Similarity: 0.9067676067352295
7
Semantic Similarity: 0.8506258726119995
8
Semantic Similarity: 0.8344841003417969
9
Semantic Similarity: 0.94203782081604
10
Semantic Similarity: 0.94203782081604
11
Semantic Similarity: 0.9385462999343872
12
Semantic Similarity: 0.9385462999343872
13
Semantic Similarity: 0.9642791748046875
14
Semantic Similarity: 0.6855964660644531
15
Semantic Similarity: 0.8984886407852173
16
Semantic Similarity: 0.8984886407852173
17
Semantic Similarity: 0.9404617547988892
18
Semantic Similarity: 0.9404617547988892
19
Semantic Similarity: 0.772111713886261
20
Semantic Similarity: 0.7605451345443726
21
Semantic Similarity: 0.8252102136611938
22
Semantic Similarity: 0.8974716663360596
23
Semantic Similarity: 0.9170470237731934
24
Semantic Similarity: 0

Semantic Similarity: 0.8422232866287231
192
Semantic Similarity: 0.8679594993591309
193
Semantic Similarity: 0.961883544921875
194
Semantic Similarity: 0.961883544921875
195
Semantic Similarity: 0.7434301376342773
196
Semantic Similarity: 0.7434301376342773
197
Semantic Similarity: 0.9531548619270325
198
Semantic Similarity: 0.9531548619270325
199
Semantic Similarity: 0.9100795984268188
200
Semantic Similarity: 0.8427411317825317
201
Semantic Similarity: 0.9221060276031494
202
Semantic Similarity: 0.9060105085372925
203
Semantic Similarity: 0.9238733053207397
204
Semantic Similarity: 0.9857629537582397
205
Semantic Similarity: 0.9923897981643677
206
Semantic Similarity: 0.9923897981643677
207
Semantic Similarity: 0.911920428276062
208
Semantic Similarity: 0.911920428276062
209
Semantic Similarity: 0.6312083005905151
210
Semantic Similarity: 0.7959184646606445
211
Semantic Similarity: 0.9056371450424194
212
Semantic Similarity: 0.9056371450424194
213
Semantic Similarity: 0.9393440485000

Semantic Similarity: 0.9228297472000122
379
Semantic Similarity: 0.9349486827850342
380
Semantic Similarity: 0.8809223771095276
381
Semantic Similarity: 0.9051936268806458
382
Semantic Similarity: 0.8266217708587646
383
Semantic Similarity: 0.9146302938461304
384
Semantic Similarity: 0.9212348461151123
385
Semantic Similarity: 0.932000994682312
386
Semantic Similarity: 0.932000994682312
387
Semantic Similarity: 0.9578737020492554
388
Semantic Similarity: 0.9578737020492554
389
Semantic Similarity: 0.9901930093765259
390
Semantic Similarity: 0.9901930093765259
391
Semantic Similarity: 0.9550637006759644
392
Semantic Similarity: 0.9550637006759644
393
Semantic Similarity: 0.8801776766777039
394
Semantic Similarity: 0.8801776766777039
395
Semantic Similarity: 0.7110204696655273
396
Semantic Similarity: 0.7110204696655273
397
Semantic Similarity: 0.6115026473999023
398
Semantic Similarity: 0.5975775122642517
399
Semantic Similarity: 0.7678027749061584
400
Semantic Similarity: 0.76780277490

Semantic Similarity: 0.8516708612442017
566
Semantic Similarity: 0.8516708612442017
567
Semantic Similarity: 0.9003729224205017
568
Semantic Similarity: 0.8166956901550293
569
Semantic Similarity: 0.8526628017425537
570
Semantic Similarity: 0.8640333414077759
571
Semantic Similarity: 0.8786101937294006
572
Semantic Similarity: 0.817780613899231
573
Semantic Similarity: 0.9214351177215576
574
Semantic Similarity: 0.9214351177215576
575
Semantic Similarity: 0.7787696123123169
576
Semantic Similarity: 0.8480383157730103
577
Semantic Similarity: 0.7917274236679077
578
Semantic Similarity: 0.765002965927124
579
Semantic Similarity: 0.9164180755615234
580
Semantic Similarity: 0.9707498550415039
581
Semantic Similarity: 0.8716698884963989
582
Semantic Similarity: 0.8807005882263184
583
Semantic Similarity: 0.982806921005249
584
Semantic Similarity: 0.8263787031173706
585
Semantic Similarity: 0.9484239220619202
586
Semantic Similarity: 0.9737815260887146
587
Semantic Similarity: 0.870542466640

In [13]:
len(embed_result)

719

In [14]:
df['embedding_match'] = embed_result

## Approach 3 : Fuzzy Matching

In [30]:
#!pip install rapidfuzz

In [15]:
from rapidfuzz import fuzz, process

def fuzzy_match(query1,query2):
    query1 = query1.lower()
    query2 = query2.lower()
    # Compute fuzzy match score
    similarity_score = fuzz.ratio(query1, query2)
    print(f"Fuzzy Similarity Score: {similarity_score}%")
    return similarity_score

In [15]:
# Example queries
query1 = "SELECT COUNT(*) FROM farm WHERE Total_Horses > 10;"
query2 = "SELECT count(*) FROM farm WHERE Total_Horses > 10;"
fuzzy_match(query1,query2)

Fuzzy Similarity Score: 100.0%


100.0

In [102]:
query1 = "SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC"
query2 = """SELECT f."Farm_ID", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;"""
fuzzy_match(query1,query2)

Fuzzy Similarity Score: 80.88235294117648%


80.88235294117648

In [16]:
fuzzy_result = []
i = 0
for index, row in df.iterrows():
    i = i+1
    spider_query = row['spider_query']
    text2sql_query = row['text2sql_query']
    print(i)
    res = fuzzy_match(spider_query,text2sql_query)
    fuzzy_result.append(res)

1
Fuzzy Similarity Score: 66.66666666666667%
2
Fuzzy Similarity Score: 64.86486486486487%
3
Fuzzy Similarity Score: 75.34246575342466%
4
Fuzzy Similarity Score: 85.9375%
5
Fuzzy Similarity Score: 84.44444444444444%
6
Fuzzy Similarity Score: 90.47619047619048%
7
Fuzzy Similarity Score: 84.55284552845528%
8
Fuzzy Similarity Score: 79.38931297709924%
9
Fuzzy Similarity Score: 77.9874213836478%
10
Fuzzy Similarity Score: 77.9874213836478%
11
Fuzzy Similarity Score: 71.02803738317758%
12
Fuzzy Similarity Score: 71.02803738317758%
13
Fuzzy Similarity Score: 91.76470588235294%
14
Fuzzy Similarity Score: 79.12087912087912%
15
Fuzzy Similarity Score: 83.96946564885496%
16
Fuzzy Similarity Score: 83.96946564885496%
17
Fuzzy Similarity Score: 92.81045751633987%
18
Fuzzy Similarity Score: 92.81045751633987%
19
Fuzzy Similarity Score: 53.99239543726235%
20
Fuzzy Similarity Score: 58.19672131147541%
21
Fuzzy Similarity Score: 58.18181818181818%
22
Fuzzy Similarity Score: 56.544502617801044%
23
Fuzzy

In [17]:
len(fuzzy_result)

719

In [18]:
df['fuzzy_match'] = fuzzy_result

In [19]:
df.head()

,db_id,spider_query,question,text2sql_query,edit_distance,embedding_match,fuzzy_match
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;",0.638889,0.770183,66.666667
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;",0.675676,0.754793,64.864865
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""...",0.493151,0.830739,75.342466
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O...",0.281250,0.887291,85.937500
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...,0.311111,0.867438,84.444444


In [20]:
import pandas as pd

# Save the DataFrame to an Excel file
########################################################################
######### CHANGE OUTPUT FOLDER HERE ####################################
########################################################################
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7-SQL-Similarity-Scores.xlsx"
df.to_excel(file_path, index=False)

print(f"DataFrame saved successfully to {file_path}")

DataFrame saved successfully to C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7-SQL-Similarity-Scores.xlsx


## Calculating Score Accuracy

In [46]:
#############################################################################
######### CHANGE OUTPUT FOLDER FILE HERE ####################################
#############################################################################
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7-SQL-Similarity-Scores.xlsx"

# Read the Excel file into a DataFrame
score_df = pd.read_excel(file_path)

# Display the first few rows of the DataFrame
score_df.head()

,db_id,spider_query,question,text2sql_query,edit_distance,embedding_match,fuzzy_match
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;",0.638889,0.770183,66.666667
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;",0.675676,0.754793,64.864865
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""...",0.493151,0.830739,75.342466
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O...",0.281250,0.887291,85.937500
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...,0.311111,0.867438,84.444444


In [47]:
edit_distance_threshold = 0.5 # if edit dist <= threshold its Match or true
embedding_match_threshold = 0.85 # if embed similairty >= thresh its a match
fuzzy_match_threshold = 75 # if fuzzy score >= thresh its a match

# Total number of elements in the column
total_elements = len(score_df['edit_distance'])

In [23]:
percentage_edit_distance = ((score_df['edit_distance'] <= edit_distance_threshold).sum() / total_elements) * 100
print(f"percentage_edit_distance: {percentage_edit_distance:.2f}%")

percentage_edit_distance: 47.29%


In [24]:
percentage_embedding_match = ((score_df['embedding_match'] >= embedding_match_threshold).sum() / total_elements) * 100
print(f"percentage_embedding_match: {percentage_embedding_match:.2f}%")

percentage_embedding_match: 64.12%


In [48]:
percentage_fuzzy_match = ((score_df['fuzzy_match'] >= fuzzy_match_threshold).sum() / total_elements) * 100
print(f"percentage_fuzzy_match: {percentage_fuzzy_match:.2f}%")

percentage_fuzzy_match: 39.08%


# ---------------------------Rough work-------------------------------
# ---------------------------Rough work-------------------------------
# ---------------------------Rough work-------------------------------
# ---------------------------Rough work-------------------------------
# ---------------------------Rough work-------------------------------

## Aproach 3 : SQL Execution

Use SQL execution to check if the two queries produce identical results on the same database.

Steps:
Execute both spider_query and text2sql_query on the target database.

Compare their outputs (e.g., row count, result set content).

This approach ensures logical correctness, but it requires a populated database.

In [50]:
"""
import sqlite3

# Example database setup
connection = sqlite3.connect(':memory:')  # In-memory database for testing
cursor = connection.cursor()
cursor.execute("CREATE TABLE farm (Total_Horses INTEGER);")
cursor.execute("INSERT INTO farm (Total_Horses) VALUES (5), (15), (20);")

# Queries
query1 = "SELECT COUNT(*) FROM farm WHERE Total_Horses > 10;"
query2 = "SELECT count(*) FROM farm WHERE Total_Horses > 10;"

def execute_query(query):
    cursor.execute(query)
    return cursor.fetchall()

# Execute and compare results
result1 = execute_query(query1)
result2 = execute_query(query2)

if result1 == result2:
    print("The queries are semantically equivalent.")
else:
    print("The queries produce different results.")
"""

'\nimport sqlite3\n\n# Example database setup\nconnection = sqlite3.connect(\':memory:\')  # In-memory database for testing\ncursor = connection.cursor()\ncursor.execute("CREATE TABLE farm (Total_Horses INTEGER);")\ncursor.execute("INSERT INTO farm (Total_Horses) VALUES (5), (15), (20);")\n\n# Queries\nquery1 = "SELECT COUNT(*) FROM farm WHERE Total_Horses > 10;"\nquery2 = "SELECT count(*) FROM farm WHERE Total_Horses > 10;"\n\ndef execute_query(query):\n    cursor.execute(query)\n    return cursor.fetchall()\n\n# Execute and compare results\nresult1 = execute_query(query1)\nresult2 = execute_query(query2)\n\nif result1 == result2:\n    print("The queries are semantically equivalent.")\nelse:\n    print("The queries produce different results.")\n'

In [ ]:
import sqlparse
import sqlite3
from typing import List

def normalize_query(query: str) -> str:
    """
    Normalize the SQL query by formatting it consistently to enable comparison.
    """
    parsed = sqlparse.format(query, keyword_case="upper", reindent=True, strip_comments=True)
    return parsed

def execute_query(query: str, db_connection: sqlite3.Connection) -> List[tuple]:
    """
    Execute the given SQL query on the database and fetch results.
    """
    cursor = db_connection.cursor()
    try:
        cursor.execute(query)
        return cursor.fetchall()
    except sqlite3.Error as e:
        print(f"Error executing query: {e}")
        return []

def are_queries_equivalent(query1: str, query2: str, db_connection: sqlite3.Connection) -> bool:
    """
    Check if two SQL queries produce the same results on the same database.
    """
    # Normalize the queries
    normalized_query1 = normalize_query(query1)
    normalized_query2 = normalize_query(query2)
    
    # Check if normalized queries are identical
    if normalized_query1 == normalized_query2:
        return True
    
    # Otherwise, compare their outputs
    result1 = execute_query(query1, db_connection)
    result2 = execute_query(query2, db_connection)
    
    return result1 == result2

    
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [21]:
import sqlparse
sqlparse.format("""select f."Farm_ID", f."Total_Horses" FROM "farm" where name='Prakhar' f Order BY f."Total_Horses" ASC;""", keyword_case="upper", reindent=True, strip_comments=True)

'SELECT f."Farm_ID",\n       f."Total_Horses"\nFROM "farm"\nWHERE name=\'Prakhar\' f\nORDER BY f."Total_Horses" ASC;'

In [ ]:
query1 = "SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC"
query2 = """SELECT f."Farm_ID", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;"""
are_same = are_queries_equivalent(query1, query2, conn)

In [ ]:
conn.close()

### Approach 4: Exact Match

In [73]:
def exact_match(query1,query2):
    if query1.strip().lower() == query2.strip().lower():
        return True
    return False

In [41]:
# Example queries
query1 = "SELECT COUNT(*) FROM farm WHERE Total_Horses > 10;"
query2 = "SELECT count(*) FROM farm WHERE Total_Horses > 10;"
exact_match(query1,query2)

True

In [42]:
query1 = "SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC"
query2 = """SELECT f."Farm_ID", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;"""
exact_match(query1,query2)

False

In [74]:
exact_result = []
i = 0
for index, row in df.iterrows():
    i = i+1
    spider_query = row['spider_query']
    text2sql_query = row['text2sql_query']
    print(i)
    res = exact_match(spider_query,text2sql_query)
    exact_result.append(res)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [75]:
calculate_accuracy(exact_result)

0.0

### Approach 5: Normalized Match

NOT IMPLEMENTED

In [76]:
import re

def normalized_match(query1, query2):
    """
    Compare two SQL queries after normalizing them.
    
    Steps:
    1. Convert queries to lowercase.
    2. Remove semicolons (;), periods (.), and quotes (' or ").
    3. Compare the resulting strings.
    
    Args:
        query1 (str): The first SQL query.
        query2 (str): The second SQL query.
    
    Returns:
        bool: True if the normalized queries are equal, False otherwise.
    """
    def normalize(query):
        # Remove comments
        query = re.sub(r'(--.*?$)|(/\*.*?\*/)', '', query, flags=re.MULTILINE)
        # Convert to lowercase
        query = query.lower()
        # Remove semicolons, periods, and quotes
        query = re.sub(r"[;.'\"]", "", query)
        # Normalize whitespace
        query = re.sub(r'\s+', ' ', query).strip()
        return query
    
    normq1 = normalize(query1)
    normq2 = normalize(query2)
    print(normq1)
    print(normq2)
    return normq1 == normq2

# Example usage
query_a = "SELECT * FROM table_name WHERE column='value';"
query_b = "select * from table_name where column='value'"

# Example usage
query_a = "SELECT * FROM table_name WHERE column='value';"
query_b = "/*comment*/  select * FROM table_name where column='value'"
print(normalized_match(query_a, query_b))  # Output: True


select * from table_name where column=value
select * from table_name where column=value
True


In [77]:
normal_result = []
i = 0
for index, row in df.iterrows():
    i = i+1
    spider_query = row['spider_query']
    text2sql_query = row['text2sql_query']
    print(i)
    res = normalized_match(spider_query,text2sql_query)
    normal_result.append(res)

1
select count(*) from farm
select count(*) from farm f
2
select count(*) from farm
select count(*) from farm
3
select total_horses from farm order by total_horses asc
select ffarm_id, ftotal_horses from farm f order by ftotal_horses asc
4
select total_horses from farm order by total_horses asc
select ffarm_id, ftotal_horses from farm f order by ftotal_horses asc
5
select hosts from farm_competition where theme != aliens
select distinct fchosts from farm_competition fc where fctheme!= aliens
6
select hosts from farm_competition where theme != aliens
select fchosts from farm_competition fc where fctheme!= aliens
7
select theme from farm_competition order by year asc
select fcyear, fctheme from farm_competition fc order by fcyear asc
8
select theme from farm_competition order by year asc
select fcyear, fctheme from farm_competition fc order by fcyear asc
9
select avg(working_horses) from farm where total_horses > 5000
select avg(fworking_horses) as average_working_horses from farm f wher

calculate_accuracy(normal_result)

In [55]:
import sqlparse
from sqlparse.sql import IdentifierList, Identifier
from sqlparse.tokens import Keyword, DML

def extract_query_components(query: str) -> dict:
    """
    Parse and extract key components from the SQL query.
    """
    parsed = sqlparse.parse(query)[0]  # Parse the SQL query
    components = {"select": [], "from": [], "where": []}
    
    current_keyword = None

    for token in parsed.tokens:
        if token.is_whitespace or token.ttype == sqlparse.tokens.Punctuation:
            continue  # Skip whitespaces and punctuation

        if token.ttype in (Keyword, DML):
            current_keyword = token.value.upper()

        if current_keyword == "SELECT" and isinstance(token, (IdentifierList, Identifier)):
            components["select"].extend([str(item).strip() for item in token.get_identifiers()])
        elif current_keyword == "FROM" and isinstance(token, Identifier):
            components["from"].append(str(token).strip())
        elif current_keyword == "WHERE" and token.ttype is None:
            components["where"].append(str(token).strip())

    return components

def are_queries_equivalent(query1: str, query2: str) -> bool:
    """
    Compare two SQL queries to check if they are logically equivalent.
    """
    # Normalize queries
    query1 = sqlparse.format(query1, keyword_case="upper", strip_comments=True)
    query2 = sqlparse.format(query2, keyword_case="upper", strip_comments=True)

    print(query1)
    print(query2)
    # Extract components from both queries
    components1 = extract_query_components(query1)
    components2 = extract_query_components(query2)
    print(components1)
    print(components2)

    # Compare the components
    return components1 == components2

# Example usage
if __name__ == "__main__":
    query1 = "SELECT id, name FROM test WHERE id = 1"
    query2 = "select name, id from test t where id = 1"

    is_equivalent = are_queries_equivalent(query1, query2)
    print(f"Are the queries equivalent? {is_equivalent}")


SELECT id, name FROM test WHERE id = 1
SELECT name, id FROM test t WHERE id = 1
{'select': ['id', 'name'], 'from': ['test'], 'where': []}
{'select': ['name', 'id'], 'from': ['test t'], 'where': []}
Are the queries equivalent? False


In [49]:
import sqlparse

def normalize_query(query):
    # Format and normalize the query
    parsed = sqlparse.format(query, reindent=True, keyword_case='upper')
    # Remove unnecessary whitespace and line breaks
    return ''.join([token.value for token in sqlparse.parse(parsed)[0].tokens if not token.is_whitespace])

def are_queries_equal(query1, query2):
    normalized_query1 = normalize_query(query1)
    print(normalized_query1)
    normalized_query2 = normalize_query(query2)
    print(normalized_query2)
    return normalized_query1 == normalized_query2

# Example usage
query1 = "SELECT * FROM users WHERE id = 1;"
query2 = "select * FROM users WHERE id=1"

if are_queries_equal(query1, query2):
    print("The SQL queries are equivalent.")
else:
    print("The SQL queries are not equivalent.")


SELECT*FROMusersWHERE id = 1;
SELECT*FROMusersWHERE id=1
The SQL queries are not equivalent.


In [46]:
import sqlparse

# Example query
query = "SELECT COUNT(*) FROM farm WHERE Total_Horses > 10;"
query = """SELECT fc.Hosts FROM "farm_competition" fc WHERE fc.Theme!= 'Aliens';"""
query = """SELECT f."Farm_ID", f."Total_Horses" FROM "farm" f ORDER BY f."Total_Horses" ASC;"""

# Parse the query
parsed = sqlparse.parse(query)[0]

# Print the parsed query
print(parsed.tokens)

[<DML 'SELECT' at 0x1B5D61EDC00>, <Whitespace ' ' at 0x1B5E7E8BB80>, <IdentifierList 'f."Far...' at 0x1B5D96AFC50>, <Whitespace ' ' at 0x1B5E7E8BDC0>, <Keyword 'FROM' at 0x1B5E7E89540>, <Whitespace ' ' at 0x1B5E7E8BD60>, <Identifier '"farm"...' at 0x1B5D96AFE50>, <Whitespace ' ' at 0x1B5E7E8B1C0>, <Keyword 'ORDER ...' at 0x1B5E7E89420>, <Whitespace ' ' at 0x1B5E7E8B160>, <Identifier 'f."Tot...' at 0x1B5D96AFD50>, <Punctuation ';' at 0x1B5E7E892A0>]


In [47]:
for token in parsed.tokens:
    if not token.is_whitespace:
        print(f"Type: {token.ttype}, Value: {token.value}")

Type: Token.Keyword.DML, Value: SELECT
Type: None, Value: f."Farm_ID", f."Total_Horses"
Type: Token.Keyword, Value: FROM
Type: None, Value: "farm" f
Type: Token.Keyword, Value: ORDER BY
Type: None, Value: f."Total_Horses" ASC
Type: Token.Punctuation, Value: ;


### Normalized match

In [43]:
import re

def normalize_sql(query):
    # Convert to lowercase
    query = query.lower()
    
    # Remove extra whitespaces
    query = re.sub(r'\s+', ' ', query).strip()
    
    # Standardize table and column aliases
    query = re.sub(r'(\s)([a-zA-Z_]\w*)(\s)(?=\.)', r' \1', query)  # removes aliasing of table/column
    
    # Remove quotes around table/column names (for simplicity, this assumes no conflicting keywords)
    query = re.sub(r'\"([^\"]+)\"', r'\1', query)
    
    # Normalize ORDER BY and GROUP BY clauses to avoid position-based differences
    query = re.sub(r'\border\sby\b.*', 'order by', query)
    query = re.sub(r'\bgroup\sby\b.*', 'group by', query)
    
    # Normalize SELECT and other SQL keywords to lowercase
    query = re.sub(r'\b(select|from|where|group by|order by|count|distinct)\b', lambda match: match.group(0).lower(), query)
    
    return query

def compare_queries(query1, query2):
    # Normalize both queries
    norm1 = normalize_sql(query1)
    norm2 = normalize_sql(query2)
    print(norm1)
    print(norm2)
    
    # Return if the normalized queries are the same
    return norm1 == norm2

# Example usage
spider_query = 'SELECT count(*) FROM farm'
text2sql_query = 'SELECT COUNT(*) FROM "farm" f'
#text2sql_query = """SELECT fc.Hosts FROM "farm_competition" fc WHERE fc.Theme!= 'Aliens';"""

normalized_match = compare_queries(spider_query, text2sql_query)
print(f"Normalized Match: {normalized_match}")


select count(*) from farm
select count(*) from farm f
Normalized Match: False


In [51]:
import sqlparse
from sqlparse.sql import Identifier, IdentifierList
from sqlparse.tokens import Keyword, DML

def remove_aliases_from_sql(sql_query):
    """
    Removes alias names from a SQL query while retaining the query's structure.
    :param sql_query: The SQL query string.
    :return: SQL query string without aliases.
    """
    parsed = sqlparse.parse(sql_query)[0]
    modified_query = []
    
    def strip_alias(identifier):
        """Strips alias from an identifier."""
        if identifier.get_real_name() != identifier.get_name():
            return identifier.get_real_name()  # Remove alias
        return identifier.get_name()
    
    for token in parsed.tokens:
        if isinstance(token, IdentifierList):
            # Process each identifier in the list
            new_identifiers = []
            for identifier in token.get_identifiers():
                if isinstance(identifier, Identifier):
                    stripped = strip_alias(identifier)
                    new_identifiers.append(stripped)
                else:
                    new_identifiers.append(str(identifier))
            modified_query.append(", ".join(new_identifiers))
        elif isinstance(token, Identifier):
            # Process individual identifiers
            stripped = strip_alias(token)
            modified_query.append(stripped)
        else:
            # Add non-identifier tokens as-is
            modified_query.append(str(token))
    
    return " ".join(modified_query)

# Example usage
sql_query = """
SELECT a.id AS user_id, b.name AS user_name, c.age FROM users a
JOIN details b ON a.id = b.user_id
JOIN info c ON b.id = c.detail_id
WHERE a.status = 'active'
"""

clean_query = remove_aliases_from_sql(sql_query)
print("Original Query:\n", sql_query)
print("\nCleaned Query:\n", clean_query)


Original Query:
 
SELECT a.id AS user_id, b.name AS user_name, c.age FROM users a
JOIN details b ON a.id = b.user_id
JOIN info c ON b.id = c.detail_id
WHERE a.status = 'active'


Cleaned Query:
 
 SELECT   id, name, age   FROM   users 
 JOIN   details   ON   a.id = b.user_id 
 JOIN   info   ON   b.id = c.detail_id 
 WHERE a.status = 'active'

